In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

In [1]:
# Load CSV files
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Training Dataset Shape:", train.shape)
print("Testing Dataset Shape:", test.shape)

train.head()

Training Dataset Shape: (10000, 785)
Testing Dataset Shape: (2000, 785)


In [1]:
plt.figure(figsize=(10,5))

for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(train.iloc[i,1:].values.reshape(28,28), cmap='gray')
    plt.title(f"Label: {train.iloc[i,0]}")
    plt.axis('off')

plt.show()

In [1]:
X = train.iloc[:,1:].values
y = train.iloc[:,0].values

X_test = test.iloc[:,1:].values
y_test = test.iloc[:,0].values

In [1]:
X = X / 255.0
X_test = X_test / 255.0

In [1]:
X_tensor = torch.tensor(X.reshape(-1, 1, 28, 28), dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.long)

X_test_tensor = torch.tensor(X_test.reshape(-1, 1, 28, 28), dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("X Tensor Shape:", X_tensor.shape)
print("X Test Tensor Shape:", X_test_tensor.shape)

X Tensor Shape: torch.Size([10000, 1, 28, 28])
X Test Tensor Shape: torch.Size([2000, 1, 28, 28])


In [1]:
val_size = int(len(X_tensor) * 0.2)
train_size = len(X_tensor) - val_size

train_dataset = TensorDataset(X_tensor[:train_size], y_tensor[:train_size])
val_dataset = TensorDataset(X_tensor[train_size:], y_tensor[train_size:])
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [1]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = ConvNet()
print(model)

ConvNet(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [1]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [1]:
num_epochs = 10
train_acc_history = []
val_acc_history = []
train_loss_history = []
val_loss_history = []

for epoch in range(1, num_epochs + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}", unit="batch")
    for batch_x, batch_y in pbar:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * batch_x.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)

        batch_acc = correct / total
        pbar.set_postfix(loss=loss.item(), acc=f"{batch_acc:.4f}")

    epoch_train_loss = running_loss / total
    epoch_train_acc = correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item() * batch_x.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == batch_y).sum().item()
            val_total += batch_y.size(0)

    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    train_loss_history.append(epoch_train_loss)
    val_loss_history.append(epoch_val_loss)
    train_acc_history.append(epoch_train_acc)
    val_acc_history.append(epoch_val_acc)

    print(f"Epoch {epoch:2d}/{num_epochs:2d} | Train Loss: {epoch_train_loss:.4f} - Train Acc: {epoch_train_acc*100:.2f}% | Val Loss: {epoch_val_loss:.4f} - Val Acc: {epoch_val_acc*100:.2f}%")

Epoch  1/10 | Train Loss: 0.8171 - Train Acc: 73.71% | Val Loss: 0.2829 - Val Acc: 91.75%
Epoch  2/10 | Train Loss: 0.2740 - Train Acc: 91.81% | Val Loss: 0.1680 - Val Acc: 94.95%
Epoch  3/10 | Train Loss: 0.1946 - Train Acc: 94.29% | Val Loss: 0.1520 - Val Acc: 95.90%
Epoch  4/10 | Train Loss: 0.1526 - Train Acc: 95.20% | Val Loss: 0.1168 - Val Acc: 96.60%
Epoch  5/10 | Train Loss: 0.1241 - Train Acc: 96.33% | Val Loss: 0.1163 - Val Acc: 96.65%
Epoch  6/10 | Train Loss: 0.1074 - Train Acc: 96.62% | Val Loss: 0.1166 - Val Acc: 96.35%
Epoch  7/10 | Train Loss: 0.0979 - Train Acc: 97.04% | Val Loss: 0.0931 - Val Acc: 96.95%
Epoch  8/10 | Train Loss: 0.0871 - Train Acc: 97.41% | Val Loss: 0.0902 - Val Acc: 97.35%
Epoch  9/10 | Train Loss: 0.0754 - Train Acc: 97.55% | Val Loss: 0.0874 - Val Acc: 97.25%
Epoch 10/10 | Train Loss: 0.0659 - Train Acc: 97.84% | Val Loss: 0.0904 - Val Acc: 97.45%


In [1]:
model.eval()
test_correct = 0
test_total = 0
all_preds = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        outputs = model(batch_x)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == batch_y).sum().item()
        test_total += batch_y.size(0)
        all_preds.extend(preds.numpy())

test_acc = test_correct / test_total
print(f"Test Accuracy: {test_acc*100:.2f}%")

Test Accuracy: 97.45%


In [1]:
plt.figure(figsize=(8,5))
plt.plot(train_acc_history, label='Training Accuracy')
plt.plot(val_acc_history, label='Validation Accuracy')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy Graph")
plt.legend()
plt.grid(True)
plt.show()

In [1]:
plt.figure(figsize=(8,5))
plt.plot(train_loss_history, label='Training Loss')
plt.plot(val_loss_history, label='Validation Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss Graph")
plt.legend()
plt.grid(True)
plt.show()

In [1]:
print("Predicted Digits:", all_preds[:20])
print("Actual Digits:   ", list(y_test_raw[:20]))

plt.figure(figsize=(10,4))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(X_test[i].reshape(28,28), cmap='gray')
    plt.title(f"Pred: {all_preds[i]} | True: {y_test_raw[i]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

Predicted Digits: [np.int64(7), np.int64(2), np.int64(1), np.int64(0), np.int64(4), np.int64(1), np.int64(4), np.int64(9), np.int64(5), np.int64(9), np.int64(0), np.int64(6), np.int64(9), np.int64(0), np.int64(1), np.int64(5), np.int64(9), np.int64(7), np.int64(3), np.int64(4)]
Actual Digits:    [np.int64(7), np.int64(2), np.int64(1), np.int64(0), np.int64(4), np.int64(1), np.int64(4), np.int64(9), np.int64(5), np.int64(9), np.int64(0), np.int64(6), np.int64(9), np.int64(0), np.int64(1), np.int64(5), np.int64(9), np.int64(7), np.int64(3), np.int64(4)]


In [1]:
torch.save(model.state_dict(), "Handwritten_Digit_Recognition_Model.pth")
pred_df = pd.DataFrame({'Actual': y_test_raw, 'Predicted': all_preds})
pred_df.to_csv("digit_predictions_results.csv", index=False)
print("Model Saved Successfully as Handwritten_Digit_Recognition_Model.pth!")
print("Saved predictions to digit_predictions_results.csv!")

Model Saved Successfully as Handwritten_Digit_Recognition_Model.pth!
Saved predictions to digit_predictions_results.csv!
